In [0]:
# ----------------------------------------------------------
# Batch Pipeline: Bronze → Silver → Gold
# ----------------------------------------------------------

from pyspark.sql.functions import col, to_date, sum, count
import re

# ----------------------------------------------------------
# Step 0: Initial Setup and Configuration
# ----------------------------------------------------------
storage_account_name = "stdataengportfolio"
storage_account_key = "<STORAGE_ACCOUNT_KEY>"

bronze_path = f"abfss://lake@{storage_account_name}.dfs.core.windows.net/bronze/salesdata/ingestion_date=2025-09-01/"
silver_path = f"abfss://lake@{storage_account_name}.dfs.core.windows.net/silver/superstore_historical/"
gold_path   = f"abfss://lake@{storage_account_name}.dfs.core.windows.net/gold/sales_aggregates/"

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

# ----------------------------------------------------------
# Step 1: Read the Bronze Layer Data
# ----------------------------------------------------------
print("➡️ Reading Bronze layer data...")
df_bronze = spark.read.csv(bronze_path, header=True, inferSchema=True)
print("✅ Bronze data loaded.")

# ----------------------------------------------------------
# Step 2: Clean & Standardize Columns
# ----------------------------------------------------------
print("➡️ Cleaning column names...")

def clean_column_name(column):
    clean_name = re.sub(r'[^a-zA-Z0-9_]', '', column.replace(' ', '_'))
    return clean_name.lower()

df_cleaned = df_bronze
for c in df_bronze.columns:
    df_cleaned = df_cleaned.withColumnRenamed(c, clean_column_name(c))

# Standardize again in case of duplicates
df_cleaned = df_cleaned.toDF(*[c.strip().lower().replace(" ", "_") for c in df_cleaned.columns])

print("✅ Columns cleaned.")

# ----------------------------------------------------------
# Step 3: Transform → Silver Layer
# ----------------------------------------------------------
print("➡️ Transforming data for Silver layer...")

df_silver = (
    df_cleaned
    .withColumn("order_date", to_date(col("order_date"), "M/d/yyyy"))
    .withColumn("ship_date", to_date(col("ship_date"), "M/d/yyyy"))
    .withColumn("total_sales", col("sales") * col("quantity"))
    .withColumn("total_profit", col("profit") * col("quantity"))
)

print("✅ Transformation complete.")

# ----------------------------------------------------------
# Step 4: Write to Silver Layer
# ----------------------------------------------------------
print(f"➡️ Writing to Silver: {silver_path}")
df_silver.write.format("delta").mode("overwrite").save(silver_path)

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS silver_historical_sales
    USING DELTA
    LOCATION '{silver_path}'
""")
print("✅ Silver table ready.")

# ----------------------------------------------------------
# Step 5: Gold Aggregation
# ----------------------------------------------------------
print("➡️ Aggregating data for Gold layer...")

df_gold = (
    df_silver.groupBy("order_date", "category", "subcategory")
    .agg(
        sum("total_sales").alias("daily_sales"),
        sum("total_profit").alias("daily_profit"),
        count("order_id").alias("orders_count")
    )
    .orderBy("order_date")
)

print("✅ Aggregation complete.")

# ----------------------------------------------------------
# Step 6: Write to Gold Layer
# ----------------------------------------------------------
print(f"➡️ Writing to Gold: {gold_path}")
df_gold.write.format("delta").mode("overwrite").save(gold_path)

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS gold_sales_aggregates
    USING DELTA
    LOCATION '{gold_path}'
""")
print("✅ Gold table ready.")


➡️ Reading Bronze layer data...
✅ Bronze data loaded.
➡️ Cleaning column names...
✅ Columns cleaned.
➡️ Transforming data for Silver layer...
✅ Transformation complete.
➡️ Writing to Silver: abfss://lake@stdataengportfolio.dfs.core.windows.net/silver/superstore_historical/
✅ Silver table ready.
➡️ Aggregating data for Gold layer...
✅ Aggregation complete.
➡️ Writing to Gold: abfss://lake@stdataengportfolio.dfs.core.windows.net/gold/sales_aggregates/
✅ Gold table ready.


In [0]:
%sql
SELECT date, SUM(total_sales) as revenue
FROM gold_sales_aggregates
GROUP BY date
ORDER BY date DESC;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7130329421307373>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'SELECT date, SUM(total_sales) as revenue\nFROM gold_sales_aggregates\nGROUP BY date\nORDER BY date DESC;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:130, in SqlMagic.sql(self, line, cell)
    126 